# 🛡️ PhishAlert — Evidencia 3

Detección de mensajes de phishing mediante procesamiento de lenguaje natural.

**Integrantes:** Marycielo Bedoya · María Pinazo · Patrick Ramírez

**Técnicas de NLP:** clasificación de textos (DistilBERT multilingüe afinado) y extracción de entidades (enlaces, correos, teléfonos, organizaciones).

Esta notebook clona el repositorio del proyecto y ejecuta todo el flujo: pruebas → datos → entrenamiento → demo web.

> Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

## 1. Preparación del entorno

In [1]:
import torch
print('GPU disponible:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU (será lento)')

GPU disponible: True - Tesla T4


In [2]:
REPO_URL = 'https://github.com/patrickram99/phishalert.git'

import os
%cd /content
if not os.path.exists('phishalert'):
    !git clone -q {REPO_URL} phishalert
%cd /content/phishalert
!git pull -q
!git log --oneline -3

/content
/content/phishalert
2822876 (HEAD -> main, origin/main, origin/HEAD) Agrega notebook de Colab para la Evidencia 3
d12868f PhishAlert: specs (Ev3 y Ev4) e implementacion de la Evidencia 3


In [3]:
!pip install -q -r requirements.txt
!python -m spacy download xx_ent_wiki_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Pruebas automatizadas

Pruebas de la extracción de entidades, el resaltado y el pipeline (no requieren modelo).

In [4]:
!python -m pytest -q

..............................                                           [100%]
30 passed in 0.11s


## 3. Datos

Descarga los datasets públicos en inglés, traduce una muestra balanceada al español, agrega el conjunto semilla en español y genera `data/phishalert.csv` con particiones train/val/test (spec 01).

In [5]:
!python scripts/get_data.py --translate-n 3000

README.md: 100% 616/616 [00:00<00:00, 3.28MB/s]

Phishing_Email.csv: downloading bytes:  58% 30.0M/52.0M [00:01<00:00, 45.7MB/s, 1.63MB/s  ]
Phishing_Email.csv: downloading bytes: 100% 30.6M/30.6M [00:01<00:00, 23.0MB/s, 2.93MB/s  ]
Phishing_Email.csv: reconstructing file: 100% 52.0M/52.0M [00:01<00:00, 39.0MB/s, 5.04MB/s  ]
Generating train split: 100% 18650/18650 [00:01<00:00, 17328.78 examples/s]
[datos] phishing_email: 17453 filas
README.md: 100% 4.98k/4.98k [00:00<00:00, 10.9MB/s]

plain_text/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/359k [00:00<?, ?B/s]
plain_text/train-00000-of-00001.parquet: downloading bytes: 100% 358k/358k [00:00<00:00, 481kB/s, 35.2kB/s  ]
plain_text/train-00000-of-00001.parquet: reconstructing file: 100% 359k/359k [00:00<00:00, 482kB/s, 35.3kB/s  ]
Generating train split: 100% 5574/5574 [00:00<00:00, 102658.11 examples/s]
[datos] sms_spam: 5129 filas
tokenizer_config.json: 100% 44.0/44.0 [00:00<00:00, 277kB/s]
source.spm: 100% 802k/802k [00

In [6]:
import pandas as pd
df = pd.read_csv('data/phishalert.csv')
display(pd.crosstab([df['lang'], df['label']], df['split'], margins=True))
pd.set_option('display.max_colwidth', 160)
df[df['lang'] == 'es'].sample(6, random_state=1)[['text', 'label', 'source']]

split       test  train   val    All
lang label                          
en   0      1396  11166  1396  13958
     1       563   4499   562   5624
es   0       152   1215   152   1519
     1       150   1207   151   1508
All         2261  18087  2261  22609

,text,label,source
21211,Hola : mi nombre es Chrissie . i 'm un estudiante de 18 años con un cuerpo fantástico que me encanta mostrar ! Acabo de empezar a trabajar en este lugar don...,1,phishing_email_traducido
22570,BBVA: detectamos un acceso no autorizado a su banca por internet. Confirme su clave digital ahora mismo en bbva-verificacion.top/login para evitar el bloque...,1,semilla_es
20696,"Está bien, llamó a mamá en lugar de divertirse",0,sms_spam_traducido
20898,"Cuánto pagó, Suganya.",0,sms_spam_traducido
21193,"SMS SERVICES. for your inclusive text credits, pls goto www.comuk.net login= 3qxj9 unuscribe with STOP, no extra charge. help 08702840625.COMUK. 220-CM2 9AE",1,sms_spam_traducido
19912,"Las Vegas - por favor lea y responda lo antes posible cada persona tiene un presupuesto de $ 190 / persona para los diferentes espectáculos / golf / spa, et...",0,phishing_email_traducido


## 4. Cargar modelo pre-entrenado desde Google Drive

En lugar de entrenar el modelo de nuevo, montaremos Google Drive para copiar los archivos del clasificador previamente descargados y guardados en tu carpeta de Drive.

In [11]:
from google.colab import drive
import os
import shutil

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta de origen en Drive y el destino local
# Nota: Debes asegurarte de que la carpeta compartida esté en tu Unidad ("My Drive") o crear un acceso directo si está compartida contigo.
# Reemplaza 'Ruta/A/Tu/Carpeta' con la ruta exacta donde pusiste los archivos dentro de tu Drive.
origen_drive = '/content/drive/MyDrive/models/phishalert'
destino_local = '/content/phishalert/models/phishalert'

os.makedirs(destino_local, exist_ok=True)

if os.path.exists(origen_drive):
    # Copiar todos los archivos de la carpeta de Drive a la carpeta local
    for item in os.listdir(origen_drive):
        s = os.path.join(origen_drive, item)
        d = os.path.join(destino_local, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print("✅ Archivos del modelo cargados exitosamente desde Google Drive.")
else:
    print(f"⚠️ No se encontró la ruta {origen_drive} en tu Google Drive. Verifica que la ruta sea correcta y que hayas montado tu cuenta correctamente.")

Mounted at /content/drive
⚠️ No se encontró la ruta /content/drive/MyDrive/models/phishalert en tu Google Drive. Verifica que la ruta sea correcta y que hayas montado tu cuenta correctamente.


## 4. Entrenamiento del clasificador

Afinado corto de `distilbert-base-multilingual-cased` (submuestra de 6000 mensajes, 1 época) para la demo al 50 % (spec 02). El entrenamiento completo corresponde a la Evidencia 4.

In [7]:
!python scripts/train.py --epochs 1 --max-train 6000 --out models/phishalert

[entrenamiento] train=6000  val=2261  base=distilbert-base-multilingual-cased
config.json: 100% 466/466 [00:00<00:00, 1.85MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 194kB/s]
vocab.txt: 100% 996k/996k [00:00<00:00, 3.97MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 5.46MB/s]

model.safetensors: downloading bytes:  45% 243M/542M [00:01<00:00, 339MB/s, 19.1MB/s  ]
model.safetensors: downloading bytes:  59% 321M/542M [00:01<00:00, 359MB/s, 26.3MB/s  ]
model.safetensors: downloading bytes:  77% 419M/542M [00:01<00:00, 241MB/s, 36.8MB/s  ]
model.safetensors: downloading bytes:  92% 499M/542M [00:02<00:00, 218MB/s, 42.9MB/s  ]
model.safetensors: reconstructing file:  87% 469M/542M [00:02<00:00, 242MB/s, 31.1MB/s  ]
model.safetensors: downloading bytes: 100% 511M/511M [00:02<00:00, 179MB/s, 44.6MB/s  ]
model.safetensors: reconstructing file: 100% 542M/542M [00:02<00:00, 190MB/s, 48.0MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 4138.15it/s]
[transformers] DistilBertF

## 5. Prueba rápida: clasificación + entidades + resaltado

In [8]:
import sys
sys.path.insert(0, '/content/phishalert/src')
from phishalert.classifier import PhishClassifier
from phishalert.pipeline import analyze
from app import EJEMPLOS

clf = PhishClassifier('models/phishalert')
for (texto,) in EJEMPLOS:
    r = analyze(texto, clf)
    print(f"[{r['label'].upper():9}] p(phishing)={r['prob_phishing']:.3f}  {texto[:70]}…")
    print('   entidades :', [(e['type'], e['text']) for e in r['entities']])
    print('   resaltados:', [(h['category'], h['text']) for h in r['highlights']])
    print()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[PHISHING ] p(phishing)=0.985  Estimado cliente: su cuenta BCP ha sido bloqueada por actividad sospec…
   entidades : [('ORG', 'BCP'), ('URL', 'http://bcp-seguridad-clientes.xyz/validar')]
   resaltados: [('amenaza', 'actividad sospechosa'), ('datos_personales', 'Verifique sus datos'), ('datos_personales', 'clave de internet'), ('urgencia', 'en las próximas 24 horas'), ('amenaza', 'de lo contrario'), ('amenaza', 'será suspendida')]

[PHISHING ] p(phishing)=0.983  Congratulations! You have been selected as the winner of a $1,000 Amaz…
   entidades : [('ORG', 'Amazon'), ('URL', 'http://bit.ly/claim-reward-now')]
   resaltados: [('premio', 'Congratulations'), ('premio', 'You have been selected'), ('premio', 'winner'), ('premio', 'gift card'), ('urgencia', 'Click here'), ('premio', 'claim your prize'), ('urgencia', 'expires today')]

[LEGITIMO ] p(phishing)=0.004  Hola equipo, les comparto el acta de la reunión del lunes. Quedamos en…
   entidades : []
   resaltados: []

[LEGITIMO ] p(phis

## 6. Demo web

Lanza la aplicación Gradio con un enlace público temporal (spec 04).

In [9]:
from app import build_demo

demo = build_demo('models/phishalert')
demo.launch(share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76b947ac4eed1c2402.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Descargar Pesos del Modelo

Ejecuta la siguiente celda para comprimir los pesos entrenados del clasificador y descargarlos automáticamente a tu computadora local.

In [10]:
import shutil
from google.colab import files

# Comprimir la carpeta del modelo entrenado
shutil.make_archive('phishalert_model', 'zip', 'models/phishalert')

# Descargar el archivo ZIP
files.download('phishalert_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>